# Step 13 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew`. You'll run the same two agents under two different `Process` strategies: `sequential` — the same fixed pipeline this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files — and `hierarchical`, where a manager decides at runtime which agent handles each task, instead of that being fixed in code.

## Learning objective

By the end of this notebook, you will:

- Understand why a second, complementary agent can produce something neither agent produces alone
- Have chained two `Task`s together with `context=[...]`, so one agent's output feeds directly into another's input
- Have run your first real `Crew` with two agents and `process=Process.sequential`
- Understand how `process=Process.hierarchical` differs: task order is still fixed, but *which* agent handles each task is a manager's runtime decision, not something wired into the code
- Have run the same two agents under a manager, via `manager_llm` (or a custom `manager_agent`)

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- The same `.env` setup as the previous steps

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

## How this works

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential`:

1. **`researcher`** is Step 09's exact same Researcher agent, unchanged.
2. **`analyst`** is a new agent with a different `role`/`goal`/`backstory` — someone who turns raw findings into a report for a specific audience, a job the Researcher was never asked to do.
3. **`research_task`** is assigned to the Researcher; **`analysis_task`** is assigned to the Analyst, with one crucial addition: `context=[research_task]`. This tells CrewAI to feed the first task's output into the second automatically — the same idea as Step 06's chain prompting, but wired declaratively instead of copy-pasted by hand.
4. **`crew.kickoff()`** runs both tasks in the order they appear in `tasks=[...]`, because `process=Process.sequential`.

Both crews below also set `tracing=True` — [Step 02](step_02_intro_to_crewai.ipynb) introduced what that gives you over `verbose=True` — so each run prints its own shareable dashboard link; `crewai_event_bus.flush()` right after `kickoff()` just makes sure that link prints before the cell finishes. Neither call changes what the agents actually do, but with two agents and two `Process` strategies to compare, a trace view earns its keep here in a way it didn't yet in Step 02.

Further down, the same two agents run again under `process=Process.hierarchical` — a different way of wiring the same pieces together.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── Crew — same sequential process as src/research_crew/crew.py ──────────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  519fc54b-e0f3-467d-8422-0d11858b0774                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: dc144927-304a-49ee-b60b-114a89a4519c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the            │
│  risk-based framework. This architecture is the cornerstone of the legislation, designed to calibrate           │
│  regulatory oversight to the potential harm posed by an AI system.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### I. The EU AI Act: Risk-Based Categories                                                                    │
│                                                                                                                 │
│  The Act categorizes AI systems into four distinct tiers based on their potential to impact the fundamental     │
│  rights, safety, and health of individuals.                                                                     │
│                                                                                                                 │
│  #### 1. Unacceptable Risk (Prohibited)                                                                         │
│  These systems are considered a clear threat to fundamental rights and are strictly **banned**.                 │
│  *   **Examples:** Social scoring systems by governments; AI used for real-time biometric identification in     │
│  public spaces by law enforcement (with narrow exceptions); AI that exploits vulnerabilities of specific        │
│  groups (age, disability); manipulative AI (subliminal techniques) that distorts behavior.                      │
│                                                                                                                 │
│  #### 2. High-Risk                                                                                              │
│  These systems are permitted but are subject to **stringent compliance requirements** before entering the EU    │
│  market. They are generally defined as those used in critical infrastructure, education, employment, essential  │
│  private/public services, and law enforcement.                                                                  │
│  *   **Examples:** AI in medical devices, critical transport infrastructure, recruitment processes (CV          │
│  filtering), or credit scoring.                                                                                 │
│                                                                                                                 │
│  #### 3. Limited Risk (Transparency)                                                                            │
│  These systems must adhere to **specific transparency obligations** to ensure users are aware they are          │
│  interacting with an AI.                                                                                        │
│  *   **Examples:** Chatbots, AI that generates or manipulates image, audio, or video content (e.g.,             │
│  Deepfakes). Users must be informed that they are interacting with a machine or that content is AI-generated.   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what obligations     │
│  apply to providers of high-risk AI systems.                                                                    │
│  Agent:                                                                                                         │
│  EU AI Act Senior Data Researcher                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Reporting Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Internal Memo: Compliance Requirements for AI Hiring Tool under the EU AI Act**                              │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Mandatory Compliance Requirements for AI-Powered Recruitment Tools                                │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, your AI-powered hiring tool is classified as **High-Risk**. Systems used in recruitment,  │
│  specifically for CV filtering, assessing candidates, and decision-making in hiring processes, are explicitly   │
│  defined as high-risk due to their significant impact on fundamental rights and professional livelihoods.       │
│                                                                                                                 │
│  ### 2. Mandatory Compliance Obligations                                                                        │
│  As a provider of a high-risk AI system, you are legally required to fulfill the following obligations before   │
│  placing your product on the EU market:                                                                         │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous, documented process to identify and mitigate risks      │
│  throughout the AI lifecycle.                                                                                   │
│  *   **Data Governance:** Ensure training and validation datasets are high-quality, representative, and free    │
│  of bias regarding the intended geographical and behavioral context.                                            │
│  *   **Technical Documentation:** Compile detailed documentation (per Annex IV) demonstrating conformity with   │
│  the Act. This must be available for market surveillance authorities.                                           │
│  *   **Record-Keeping:** Implement automatic event logging to ensure full traceability of the AI’s              │
│  decision-making process.                                                                                       │
│  *   **Transparency & Information:** Provide clear instructions to clients (deployers) regarding the system’s   │
│  limitations, capabilities, and necessary human oversight.                                                      │
│  *   **Human Oversight:** Integrate features that allow human operators to effectively supervise, interpret,    │
│  override, or reverse AI outputs.                                                                               │
│  *   **Robustness & Cybersecurity:** Ensure the system is resilient against unauthorized manipulation and       │
│  maintains consistent accuracy throughout its lifecycle

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  EU AI Act Reporting Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 0ca1da29-459c-4d2f-a5e0-ba3d147d6bca                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/0ca1da29-459c-4d2f-a5e0-ba3d147d6bca?access_code=TRA │
│ CE-5402b9c06c                                                                                                   │
│ 🔑 Access Code: TRACE-5402b9c06c                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  519fc54b-e0f3-467d-8422-0d11858b0774                                                                           │
│  Final Output: **Internal Memo: Compliance Requirements for AI Hiring Tool under the EU AI Act**                │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Date:** May 22, 2024                                                                                         │
│  **Subject:** Mandatory Compliance Requirements for AI-Powered Recruitment Tools                                │
│                                                                                                                 │
│  ### 1. Classification                                                                                          │
│  Under the EU AI Act, your AI-powered hiring tool is classified as **High-Risk**. Systems used in recruitment,  │
│  specifically for CV filtering, assessing candidates, and decision-making in hiring processes, are explicitly   │
│  defined as high-risk due to their significant impact on fundamental rights and professional livelihoods.       │
│                                                                                                                 │
│  ### 2. Mandatory Compliance Obligations                                                                        │
│  As a provider of a high-risk AI system, you are legally required to fulfill the following obligations before   │
│  placing your product on the EU market:                                                                         │
│                                                                                                                 │
│  *   **Risk Management System:** Establish a continuous, documented process to identify and mitigate risks      │
│  throughout the AI lifecycle.                                                                                   │
│  *   **Data Governance:** Ensure training and validation datasets are high-quality, representative, and free    │
│  of bias regarding the intended geographical and behavioral context.                                            │
│  *   **Technical Documentation:** Compile detailed documentation (per Annex IV) demonstrating conformity with   │
│  the Act. This must be available for market surveillance authorities.                                           │
│  *   **Record-Keeping:** Implement automatic event logging to ensure full traceability of the AI’s              │
│  decision-making process.                                                                                       │
│  *   **Transparency & Information:** Provide clear instructions to clients (deployers) regarding the system’s   │
│  limitations, capabilities, and necessary human oversight.                                                      │
│  *   **Human Oversight:** Integrate features that allow human operators to effectively supervise, interpret,    │
│  override, or reverse AI outputs.                     

=== Researcher output ===
As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the risk-based framework. This architecture is the cornerstone of the legislation, designed to calibrate regulatory oversight to the potential harm posed by an AI system.

---

### I. The EU AI Act: Risk-Based Categories

The Act categorizes AI systems into four distinct tiers based on their potential to impact the fundamental rights, safety, and health of individuals.

#### 1. Unacceptable Risk (Prohibited)
These systems are considered a clear threat to fundamental rights and are strictly **banned**.
*   **Examples:** Social scoring systems by governments; AI used for real-time biometric identification in public spaces by law enforcement (with narrow exceptions); AI that exploits vulnerabilities of specific groups (age, disability); manipulative AI (subliminal techniques) that distorts behavior.

#### 2. High-Risk
These systems are permitted but are subject to **st

### A non-sequential pattern: manager delegation (`Process.hierarchical`)

Above, each `Task` is hard-wired to one `Agent` via `agent=...`, and `crew.kickoff()` runs them in the order they appear in `tasks=[...]`. `Process.hierarchical` removes that fixed wiring: the tasks below don't get an `agent=` at all. Instead, a **manager** — either an `Agent` CrewAI builds for you from `manager_llm`, or your own `Agent` passed as `manager_agent` — reads each task and decides, at that moment, which of the crew's agents is best suited, then delegates to them via a tool call.

Two things worth being precise about, since "hierarchical" invites the wrong mental model:

- **Task order still isn't dynamic.** The `for` loop over `tasks=[...]` doesn't change — task 1 still runs before task 2. What's dynamic is *who* does each task, decided fresh every run, not *when* it happens.
- **The manager is not one of your agents.** It's a separate role that does no task work itself — it only reads tasks and delegates. CrewAI enforces this: a `manager_agent` may not also appear in `agents=[...]`, and it may not be given `tools=` (only the agents it delegates to can have tools).

The `researcher` and `analyst` from above are reused unchanged below — same `role`/`goal`/`backstory`, no code changes to either agent. Only the `Crew`'s wiring is different.

In [2]:
# ── Same two tasks, but neither gets an `agent=` — the manager decides who does what ──
research_task_h = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
)

analysis_task_h = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
)

# ── Crew — same two agents as coworkers, orchestrated by a manager instead of fixed code ──
hierarchical_crew = Crew(
    agents=[researcher, analyst],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task_h, analysis_task_h],
    process=Process.hierarchical,
    manager_llm=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"),
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes — same as
    # the sequential crew above, so you can compare both runs side by side.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result_h = hierarchical_crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Research task, as delegated by the manager ===")
print(research_task_h.output.raw)
print("\n=== Analysis task, as delegated by the manager ===")
print(analysis_task_h.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  4efd235a-3ed0-4401-bde7-69d2e02f437a                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 76d11f4a-a395-47a5-90a5-4458ac58e1a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'EU AI Act Senior Data Researcher', 'context': "I need a comprehensive explanation of the   │
│  EU AI Act's risk-based framework. Specifically:\n1. Explain the four risk categories: Unacceptabl...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Create a structured, detailed explanation of the EU AI Act's four risk categories and the specific       │
│  obligations for providers of high-risk AI systems.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Internal Report: Compliance Requirements for AI-Powered Hiring Tool                                        │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** Management                                                                                           │
│  **Subject:** EU AI Act Compliance Strategy for Recruitment Systems                                             │
│                                                                                                                 │
│  Our current AI-powered hiring tool (CV screening and candidate evaluation) is classified as a **High-Risk AI   │
│  system** under the EU AI Act. Because our tool impacts fundamental rights in employment, we are subject to     │
│  the full suite of mandatory requirements outlined in the regulation.                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Mandatory Compliance Timeline                                                                           │
│  The EU AI Act entered into force in August 2024. For our classification, the deadline for full compliance is   │
│  **August 2027 (36 months after entry into force).**                                                            │
│                                                                                                                 │
│  While we have time, the complexity of these requirements necessitates a phased approach to our product         │
│  roadmap:                                                                                                       │
│                                                                                                                 │
│  *   **Phase 1 (Preparation, Now – Late 2025):** Establish our internal **Quality Management System (QMS)**     │
│  and finalize data governance policies to ensure training, validation, and testing datasets are representative  │
│  and bias-free.                                                                                                 │
│  *   **Phase 2 (Implementation, 2026):** Integrate technical safeguards, specifically **logging                 │
│  (traceability)** features and **human-in-the-loop** overrides. Initiate stress testing for model robustness    │
│  and accuracy.                                                                                                  │
│  *   **Phase 3 (Conformity, Early 2027):** Execute the formal **Conformity Assessment**, finalize the EU        │
│  Declaration of Conformity, and prepare to affix the **CE marking**.                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 9933c9a1-d57f-40b9-8120-1eff4e5c33a4                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/9933c9a1-d57f-40b9-8120-1eff4e5c33a4?access_code=TRA │
│ CE-b6847a6ec7                                                                                                   │
│ 🔑 Access Code: TRACE-b6847a6ec7                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  4efd235a-3ed0-4401-bde7-69d2e02f437a                                                                           │
│  Final Output: ### Internal Report: Compliance Requirements for AI-Powered Hiring Tool                          │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **From:** Management                                                                                           │
│  **Subject:** EU AI Act Compliance Strategy for Recruitment Systems                                             │
│                                                                                                                 │
│  Our current AI-powered hiring tool (CV screening and candidate evaluation) is classified as a **High-Risk AI   │
│  system** under the EU AI Act. Because our tool impacts fundamental rights in employment, we are subject to     │
│  the full suite of mandatory requirements outlined in the regulation.                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Mandatory Compliance Timeline                                                                           │
│  The EU AI Act entered into force in August 2024. For our classification, the deadline for full compliance is   │
│  **August 2027 (36 months after entry into force).**                                                            │
│                                                                                                                 │
│  While we have time, the complexity of these requirements necessitates a phased approach to our product         │
│  roadmap:                                                                                                       │
│                                                                                                                 │
│  *   **Phase 1 (Preparation, Now – Late 2025):** Establish our internal **Quality Management System (QMS)**     │
│  and finalize data governance policies to ensure training, validation, and testing datasets are representative  │
│  and bias-free.                                                                                                 │
│  *   **Phase 2 (Implementation, 2026):** Integrate technical safeguards, specifically **logging                 │
│  (traceability)** features and **human-in-the-loop** overrides. Initiate stress testing for model robustness    │
│  and accuracy.                                                                                                  │
│  *   **Phase 3 (Conformity, Early 2027):** Execute the formal **Conformity Assessment**, finalize the EU        │
│  Declaration of Conformity, and prepare to affix the **CE marking**.                                            │
│                                                       

=== Research task, as delegated by the manager ===
### The EU AI Act: A Risk-Based Framework

The EU AI Act employs a tiered, risk-based approach to regulation, where the intensity of legal obligations is proportional to the level of risk the AI system poses to the safety and fundamental rights of individuals.

---

### I. The Four Risk-Based Categories

1.  **Unacceptable Risk (Prohibited):**
    These systems are deemed a clear threat to fundamental rights and EU values. Their development, placement on the market, and use are strictly prohibited. This includes:
    *   **Cognitive behavioral manipulation:** Using subliminal techniques to distort behavior.
    *   **Social scoring:** AI used by public authorities to classify people based on social behavior or personality traits.
    *   **Biometric identification:** Real-time remote biometric identification in publicly accessible spaces for law enforcement (subject to extremely limited and narrow exceptions).
    *   **Predictive poli

### Comparing the two runs directly

Same job, same two agents, two different `Process` strategies — here's both final reports next to each other, so you're not scrolling between two cells to compare them:

In [3]:
print("=" * 80)
print("SEQUENTIAL — Analyst's report (agent= fixed in code)")
print("=" * 80)
print(analysis_task.output.raw)

print("\n" + "=" * 80)
print("HIERARCHICAL — Analyst's report (agent chosen by the manager)")
print("=" * 80)
print(analysis_task_h.output.raw)

SEQUENTIAL — Analyst's report (agent= fixed in code)
**Internal Memo: Compliance Requirements for AI Hiring Tool under the EU AI Act**

**To:** Product Team
**From:** EU AI Act Reporting Analyst
**Date:** May 22, 2024
**Subject:** Mandatory Compliance Requirements for AI-Powered Recruitment Tools

### 1. Classification
Under the EU AI Act, your AI-powered hiring tool is classified as **High-Risk**. Systems used in recruitment, specifically for CV filtering, assessing candidates, and decision-making in hiring processes, are explicitly defined as high-risk due to their significant impact on fundamental rights and professional livelihoods.

### 2. Mandatory Compliance Obligations
As a provider of a high-risk AI system, you are legally required to fulfill the following obligations before placing your product on the EU market:

*   **Risk Management System:** Establish a continuous, documented process to identify and mitigate risks throughout the AI lifecycle.
*   **Data Governance:** Ensur

## Your task

1. Run the sequential cell. Compare the Analyst's report to Step 09's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Run the hierarchical cell, then the comparison cell right after it. Judge the two runs on three separate axes — they don't move together:
   - **Content** — read both reports in the comparison cell's output. Different wording aside, is there an actual difference in what's covered or recommended, or are they substantively the same?
   - **Delegation path** — open both trace links (`app.crewai.com`, no signup needed) and compare timelines, or scan the hierarchical cell's verbose log for "Delegate work to coworker" / "Ask question to coworker" tool calls. Did the manager route to the Researcher for the first task and the Analyst for the second — matching the sequential order — or something else?
   - **Cost & latency** — same two trace timelines: count LLM calls and total run time. `Process.hierarchical` always adds at least one extra call (the manager's own reasoning) on top of whichever agent it delegates to.
   
   For two agents this cleanly separated, the honest answer is usually: same content, same delegation order, at a higher cost — see Shortcomings below for when that stops being true.

4. Now swap in your own team's topic — keep the Researcher agent from Step 09, and give it a second, genuinely complementary role and task suited to your topic.

5. This is where your project shifts from guided exercises to your own design: use everything from Steps 03–13 to build and document your own agent. Fill in `REPORT.md` — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full report structure and what's expected. Make sure the **Sprint Progression** table has all five rows filled in — it's the fastest way for your reader to see the arc before they read the rest of the report.

## Shortcomings

The sequential pipeline is fixed: the Researcher always runs first, the Analyst always runs second, and the order never changes based on what either agent actually finds. `Process.hierarchical` fixes *who* handles each task, not *when* — the task order in `tasks=[...]` is still fixed, and it costs more: every task now takes an extra LLM call (the manager's own reasoning) on top of whichever agent it delegates to, and the manager's choice of delegate isn't guaranteed to be consistent between runs. For two agents with clearly non-overlapping roles like these, that extra machinery mostly buys you nothing — it starts to pay off once you have enough agents, or similar-enough roles, that hard-coding `agent=...` on every task stops being obvious.

Neither `Crew` above uses any of the tools/MCP/RAG grounding from Steps 10–12 — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

This is the last step in the exercise series. From here, the main [README's use-case table](../../README.md#use-cases-to-pick-from) and `REPORT.md` ask you to step back and judge, for your own topic, which combination of everything covered — single vs. multi-agent, sequential vs. hierarchical, tools, MCP, RAG — is actually worth the added complexity.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered briefly in Step 02

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. The hierarchical crew is the more natural fit for this: add the agent to `agents=[...]` and a `Task` for it to `tasks=[...]`, but don't pin it with `agent=...` — let the manager decide on its own whether and when to bring it in, rather than wiring it into a fixed spot in the pipeline like you'd have to for the sequential version. Does the output meaningfully change? If it doesn't, ask yourself why.

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.